In [23]:
using LowLevelFEM, LinearAlgebra
using IterativeSolvers, Preconditioners

[ Info: Precompiling LowLevelFEMIterativeSolversExt [55978d5f-b2ed-541f-a526-69ba3ceeb2d1] (caches not reused: 6 for different dependency version already loaded, 3 for different Julia build configuration)
Precompiling packages...
   3.6 s  ✓ LowLevelFEMIterativeSolversExt (serial)
  1 dependency successfully precompiled in 6 seconds
[ Info: Precompiling LowLevelFEMPreconditionersExt [731aa7ef-afda-5ca4-bfd0-d21a0237399b] (caches not reused: 7 for different dependency version already loaded, 2 for different Julia build configuration)
Precompiling packages...
   3.5 s  ✓ LowLevelFEMPreconditionersExt (serial)
  1 dependency successfully precompiled in 4 seconds


# Axisymmetric Linear Elasticity

This example demonstrates the solution of a two-dimensional axisymmetric
linear elasticity problem using LowLevelFEM.

The problem is solved in two different ways:

1. using the built-in axisymmetric elasticity solver,
2. using the weak-form DSL.

The two solutions are then compared to verify the correctness of the
DSL implementation.

In [24]:
structured_rect_mesh(x0=20, lx=80, ly=20, n=80, order=2)
mat = Material("body");

## Geometry and material

A quadratic structured mesh is generated for a hollow cylinder.
The inner radius avoids the singularity associated with the axis of
revolution.

Linear isotropic elastic material properties are assigned to the body.

## Built-in axisymmetric formulation

First, the problem is solved using the dedicated axisymmetric elasticity
implementation available in LowLevelFEM.

The right boundary is subjected to a radial traction, while the lower-left
corner is constrained to remove rigid body motion.

In [25]:
prob = Problem([mat], type=:AxiSymmetric)

ld = load("left", fx=1);

In [26]:
bc = BoundaryCondition("leftbottom", uy=0);

In [27]:
u1 = solveDisplacement(prob, load=[ld], support=[bc]);

In [49]:
@time K1 = stiffnessMatrix(prob)
K1[:, :]

  1.641320 seconds (16.03 M allocations: 2.586 GiB, 21.45% gc time)


206402×206402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6565406 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⠿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠒⠦⢤⣀⡀⠀⎥
⎢⣿⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⎥
⎢⣿⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [29]:
S1 = solveStress(u1);

In [30]:
showDoFResults(u1, name="u1", visible=true, factor=1e5)
showStressResults(S1, name="σ1 - von Mises");

## Reference solution

The displacement and von Mises stress obtained from the built-in
axisymmetric formulation will be used as the reference solution.

## Weak-form DSL formulation

The same problem is assembled using the LowLevelFEM weak-form DSL.

The axisymmetric strain-displacement operator is written directly from
its mathematical definition.

In [31]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u);

In [32]:
r = ScalarField(Pu, "body", (x, y, z)->x);

### Axisymmetric strain operator

The strain vector is written as

$$
\varepsilon =
\begin{bmatrix}
\partial u_r/\partial r \\
u_r/r \\
\partial u_z/\partial z \\
\partial u_r/\partial z + \partial u_z/\partial r
\end{bmatrix},
$$

which is represented as

$$
\varepsilon = A_1 \nabla^s u + A_2 u.
$$

This decomposition allows the operator to be expressed naturally in the DSL.

In [33]:
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]

4×3 Matrix{Int64}:
 1  0  0
 0  0  0
 0  1  0
 0  0  1

In [34]:
A2 = [0 0; 1/r 0; 0 0; 0 0];

In [35]:
B = full(A1 ⋅ SymGrad(Pu)) + reduced(A2 ⋅ Pu);

In [36]:
E = mat.E
ν = mat.ν

D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

### System assembly

The global stiffness matrix is assembled from the weak form

$$
K = \int_\Omega B^\mathrm{T} D B \; 2\pi r \; d\Omega.
$$

The factor $2\pi r$ accounts for the axisymmetric volume measure.

In [37]:
GC.gc()

In [51]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r))
K2[:, :]

  2.336107 seconds (541.77 k allocations: 378.486 MiB, 5.90% gc time)


206402×206402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6566377 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⠿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠒⠦⢤⣀⡀⠀⎥
⎢⣿⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⎥
⎢⣿⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [39]:
f2 = ∫(Pu ⋅ [1, 0] * (2π*r), Γ="left");

In [40]:
using Profile

# Compile and initialize all relevant methods.
K2_warmup = ∫(B' ⋅ D ⋅ B * (2π * r))

GC.gc()
Profile.clear()

@time begin
    @profile K2_profiled = ∫(B' ⋅ D ⋅ B * (2π * r))
end

Profile.print(
    format=:flat,
    sortedby=:count,
    mincount=10,
    C=false
)

  2.855378 seconds (541.81 k allocations: 378.488 MiB, 0.47% gc time)
 Count  Overhead File                    Line Function
 =====  ======== ====                    ==== ========
    10         0 @Base/]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.13.3%2B0.x64.linux.gnu/share/julia/base/dict.jl\dict.jl]8;;\           134 isslotempty
    10         0 @LowLevelFEM/]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield/coefficients.jl\…field/coefficients.jl]8;;\  700 _eval_coefficient_chain_at_gp!(ws…
    10         0 @Base/]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.13.3%2B0.x64.linux.gnu/share/julia/base/broadcast.jl\broadcast.jl]8;;\      907 materialize!
    10         0 @Base/]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.13.3%2B0.x64.linux.gnu/share/julia/base/broadcast.jl\broadcast.jl]8;;\      910 materialize!
    10         0 @Base/]8;;file://perebal-La

In [41]:
Profile.print(
    format=:tree,
    sortedby=:count,
    mincount=10,
    C=false
)

Overhead ╎ [+additional indent] Count File:Line  Function
    ╎212  ]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/eventloop.jl\@IJulia]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/eventloop.jl\/…/eventloop.jl:71]8;;\  (::IJulia.var"#waitloop##2#waitloop##3"{IJ…
    ╎ 212  ]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/eventloop.jl\@IJulia]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/eventloop.jl\/…/eventloop.jl:26]8;;\  eventloop(socket::ZMQ.Socket, kernel::IJu…
    ╎  212  ]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/execute_request.jl\@IJulia]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/execute_request.jl\/…te_request.jl:129]8;;\  execute_request(socket::ZMQ.Socket, ker…
    ╎   212  ]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-

## Solution

The linear system assembled from the DSL is solved using the same boundary
conditions as the reference formulation.

In [42]:
@time u2 = solveField(K2, f2, support=[bc]);

  8.059833 seconds (12.19 M allocations: 1.883 GiB, 7.60% gc time, 47.96% compilation time: 100% of which was recompilation)


In [43]:
@time u2 = solveField(Symmetric(K2), f2, support=[bc]);

  4.398884 seconds (3.16 M allocations: 1.180 GiB, 33.20% gc time, 22.21% compilation time: 100% of which was recompilation)


In [44]:
@time u3 = solveField(Symmetric(K2), f2, support=[bc], solver=:cg);

 23.968219 seconds (2.96 M allocations: 902.682 MiB, 0.23% gc time, 9.57% compilation time)


In [47]:
@time u3 = solveField(Symmetric(K2), f2, support=[bc], solver=:cg, solveroptions=(reltol=1e-4, maxiter=1000));

 14.474392 seconds (437.42 k allocations: 753.120 MiB, 1.25% gc time)


In [ ]:
showDoFResults(u2, name="u2", visible=true, factor=1e5);

## Stress recovery

The strain components are computed from the displacement field,
followed by Hooke's law to obtain the stress tensor.

Finally, the von Mises equivalent stress is evaluated.

In [ ]:
εr = ∂x(u2[1])
εφ = u2[1] / r
εz = ∂y(u2[2])
γrz = ∂y(u2[1]) + ∂x(u2[2]);

In [ ]:
ε = [εr, εφ, εz, γrz];

In [ ]:
σ = D * ε;

In [ ]:
σr = σ[1]
σφ = σ[2]
σz = σ[3]
τrz = σ[4];

In [ ]:
σeqv2 = √(((σr - σφ)^2 + (σφ - σz)^2 + (σz - σr)^2) / 2 + 3τrz^2);

In [ ]:
showElementResults(σeqv2, name="σ2 - von Mises");

## Verification

The displacement and stress fields obtained from the weak-form DSL agree
with those produced by the built-in axisymmetric formulation, confirming
the correctness of the DSL implementation.

In [ ]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
